# 13 - Human in the Loop (Approvals)

## Purpose

This notebook demonstrates a LangChain DeepAgents tutorial episode. It walks through the key concepts and code patterns for building, configuring, and operating AI agents with the `deepagents` framework.

## What You'll Learn

- How deep agents differ from regular LangChain agents
- The built-in tool suite every agent receives automatically
- How to configure models, backends, permissions, and memory
- Practical patterns for tools, planning, memory, and handoffs

## Tutorial Series Context

This is Episode 13-human_in_the_loop of the LangChain DeepAgents YouTube tutorial series. Each episode focuses on a specific aspect of agent development, from basic introduction to advanced patterns like subagents, memory, permissions, and skills.

## How to Use This Notebook

1. **Run cells top-to-bottom**: The code is designed to execute in order, with each section building on the previous one.
2. **Observe tool calls**: Watch for `tool_calls` in message outputs - this shows which agents/tools are being used.
3. **Modify and experiment**: The code is intentionally kept identical to the original Python scripts. Feel free to modify parameters, models, or prompts to see how behavior changes.
4. **Refer to comments**: Each code section has accompanying markdown explanations of what the code does and why.

## Key Concepts Covered in This Episode

- interrupt_on pauses agent BEFORE risky tool runs
- Four decision types: approve, reject, edit, respond
- Approve: tool runs with original arguments
- Reject: tool skipped, message goes back to model
- Edit: run tool with YOUR supplied edited arguments
- Respond: skip tool, feed your own text as result
- **interrupt** / Command(resume=...) flow from episode 13


# Overview: 13-human_in_the_loop.ipynb

This notebook converts the Python tutorial `13-human_in_the_loop.ipynb` into an interactive Jupyter Notebook format, preserving all code examples and explanations exactly as they appear in the original tutorial scripts.

## Notebook Structure

Each logical section of the original Python script has been split into its own code cell. Each code cell is preceded by a markdown cell containing:

- **A short heading** describing what that section does
- **1–3 sentences** explaining the purpose and key concepts of that section

## How to Navigate

- **Run cells in order**: The tutorial was designed to be executed top-to-bottom. Each section builds on the previous one.
- **Observe tool calls**: Watch the output for `tool_calls` entries - this shows which built-in or tools the agent invokes.
- **Read the markdown comments**: Each code section has a preceding explanation of what the code does and why it's structured that way.
- **Experiment freely**: The original `.py` files are unchanged. Feel free to modify, extend, or experiment with the code in this notebook.

## Educational Value

This notebook serves as both a working tutorial replication and a learning resource. By running each cell and reading the accompanying explanations, you'll gain hands-on experience with:

- LangChain DeepAgents framework architecture
- Backend configuration (virtual vs real filesystem)
- Tool definition and registration
- Planning, memory, permissions, and context management
- Subagent delegation and structured output
- Streaming modes and human-in-the-loop patterns

## Tips for Getting the Most Out of This Notebook

1. **Don't skip the markdown cells**: The educational content between code sections is essential for understanding the "why" behind the code.
2. **Re-run after modifications**: After experimenting with changes, re-run all cells from the top to ensure everything still works together.
3. **Refer to the original Python scripts**: The `.py` files in the same directory are unchanged and can be referenced for the exact original code.
4. **Use rich output**: This notebook uses Rich formatting for colorful console output - pay attention to color-coded hints and status messages.

---

_Notebook automatically generated from `13-human_in_the_loop.ipynb`. Original Python scripts preserved unchanged._


## Section 2: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section sets up the imports and environment configurati
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
"""
13 - Human in the Loop (Approvals)
==================================
- `interrupt_on` pauses the agent BEFORE a risky tool runs
- The run stops and returns `result["__interrupt__"]` describing what it wants
- You resume with `Command(resume={"decisions": [...]})`
- Requires a checkpointer — the paused run has to be stored somewhere
Run:  python 13-human_in_the_loop.py
"""

## Section 4: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 1: Imports and setup
import os

from deepagents import create_deep_agent
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command  # carries the human's decision back in
from rich import print

load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")


def text_of(message) -> str:
    """Normalize message content across providers (str vs content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

## Section 6: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 2: A tool with real-world consequences
def send_email(to: str, body: str) -> str:
    """Send an email to a recipient.

    Args:
        to: The recipient's email address.
        body: The body text of the email.
    """
    return f"Email delivered to {to}"

## Section 8: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements gate that tool behind human approval
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 3: Gate that tool behind human approval
# interrupt_on maps tool name -> True (all four decision types allowed) or an
# InterruptOnConfig dict such as {"allowed_decisions": ["approve", "reject"]}.
agent = create_deep_agent(
    model=MODEL,
    tools=[send_email],
    interrupt_on={"send_email": True},
    checkpointer=InMemorySaver(),  # REQUIRED: the pause must be persisted
    system_prompt="You are an assistant. Use send_email when asked. Be brief.",
)

## Section 10: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements a helper that runs, inspects the pau
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 4: A helper that runs, inspects the pause, and resumes
def run_with_decision(decision: dict, thread_id: str) -> None:
    """Start a run, show what the agent wants to do, then apply `decision`.

    Each call uses a FRESH thread_id — reusing one would resume the previous,
    already-resolved run instead of triggering a new interrupt.
    """
    config = {"configurable": {"thread_id": thread_id}}
    request = "Email alice@example.com saying the report is ready."
    result = agent.invoke({"messages": [{"role": "user", "content": request}]}, config=config)

    # The pause shows up as "__interrupt__" in the returned state.
    interrupts = result.get("__interrupt__")
    if not interrupts:
        print("  [yellow]No interrupt fired — the agent never called the tool.[/yellow]")
        return

    for action in interrupts[0].value["action_requests"]:
        print(f"  [bold]paused before:[/bold] [green]{action['name']}[/green]({action['args']})")

    # Resume. "decisions" is a LIST, one entry per pending action request.
    print(f"  [bold]human decides:[/bold] [magenta]{decision['type']}[/magenta]")
    resumed = agent.invoke(Command(resume={"decisions": [decision]}), config=config)
    print(f"  [bold]outcome:[/bold] {text_of(resumed['messages'][-1]).strip()}\n")

## Section 12: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements approve — the tool runs with the ori
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 5: Approve — the tool runs with the original arguments
print("[bold cyan]1. APPROVE[/bold cyan]")
run_with_decision({"type": "approve"}, thread_id="approve-run")

## Section 14: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements reject — the tool is skipped and you
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 6: Reject — the tool is skipped and your message goes back to the model
print("[bold cyan]2. REJECT[/bold cyan]")
run_with_decision(
    {"type": "reject", "message": "Do not email anyone. Draft it for me instead."},
    thread_id="reject-run",
)

## Section 16: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 7: The other two decision types
# Same mechanism, different payloads — swap either into run_with_decision():
#
#   edit    -> run the tool, but with arguments YOU supply:
#              {"type": "edit",
#               "edited_action": {"name": "send_email",
#                                 "args": {"to": "bob@example.com", "body": "..."}}}
#
#   respond -> skip the tool and feed your own text back as its result:
#              {"type": "respond", "message": "Already sent it myself."}
print("[dim]See Step 7 in the source for the 'edit' and 'respond' payloads.[/dim]")

# %%